## TrainTest split check

### By: Carlos Javier Palacios Sanchez

### Date: 06/09/2026

### Description:

Requerimiento
Modificar `train_pipeline.py` para verificar la separación entre los conjuntos train y
test. El objetivo es evitar fuga de información (*data leakage*) y comprobar que ambos
conjuntos representan la distribución esperada del problema.

El proceso debe incluir:

- Resultados de las validaciones de la separación.
- Una advertencia clara o un error controlado cuando se detecte un problema.
- Una función testeable e independiente, por ejemplo `validate_train_test_split()`.

Se pueden utilizar herramientas como Evidently o DeepChecks.

Entregables:

- `train_pipeline.py` con los checks integrados.
- Pruebas unitarias para casos válidos e inválidos.

---

## Estrategia

### Dos preguntas distintas

El requerimiento pide comprobar dos cosas que suelen confundirse:

1. **¿Hay fuga de información?** ¿Alguna fila del test la vio el modelo al entrenar?
   Si la respuesta es sí, el test no mide generalización y las métricas están infladas.
   Esto es un **defecto**: no hay nada que interpretar, la evaluación es inválida.
2. **¿Los dos conjuntos representan el mismo problema?** ¿La partición dejó el test con
   otra prevalencia, otra distribución de edad, más faltantes? Aquí no hay nada roto,
   pero el test puede haber dejado de ser representativo. Esto es un **riesgo**, y quien
   lo interpreta necesita saberlo, no que el pipeline aborte.

Por eso las comprobaciones tienen dos severidades: **error** detiene el pipeline,
**advertencia** se registra y deja continuar. La bandera `--estricto` permite elevar
las advertencias a errores cuando el contexto lo exige (por ejemplo, en CI).

### Por qué no Evidently ni DeepChecks

Ambas son buenas herramientas y resuelven este problema con reportes HTML muy
completos. No las usé por la misma razón que descarté Great Expectations en el
entregable anterior: para un dataset de 480 filas y 26 atributos, arrastran decenas de
dependencias y un modelo de configuración propio para producir nueve comprobaciones que
caben en un módulo con `scipy`, que ya es dependencia del proyecto.

Hay además una razón de fondo: al escribir los checks a mano, cada umbral queda
justificado y cada mensaje de error dice exactamente qué se rompió. Un reporte genérico
de drift no distingue entre "esta columna difiere porque la partición fue mala" y "esta
columna difiere porque tiene tres valores distintos y el azar los repartió así".

### Las nueve comprobaciones

| Comprobación | Severidad si falla | Qué detecta |
|---|---|---|
| tamaño mínimo | error | un conjunto tan pequeño que las métricas no significan nada |
| columnas | error | esquemas distintos entre train y test |
| índices disjuntos | error | fuga directa: la misma fila en ambos conjuntos |
| filas duplicadas entre conjuntos | error | fuga encubierta: la misma fila con otro índice |
| cobertura de clases | error | falta una clase en train o en test |
| proporción de test | advertencia | el reparto se desvía de lo configurado |
| estratificación | advertencia | prevalencia distinta entre train y test |
| distribución de atributos (KS) | advertencia | algún atributo se distribuye distinto |
| faltantes comparables | advertencia | patrón de nulos distinto entre conjuntos |

## 1. Configuración

In [ ]:
import subprocess
import sys
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd


def localizar_raiz() -> Path:
    """Sube por el árbol de directorios hasta encontrar el pyproject.toml."""
    actual = Path.cwd().resolve()
    for candidato in (actual, *actual.parents):
        if (candidato / "pyproject.toml").is_file():
            return candidato
    raise FileNotFoundError("No se encontró la raíz del proyecto")


RAIZ = localizar_raiz()
SCRIPT = RAIZ / "src" / "pipelines" / "training_pipeline" / "train_pipeline.py"
PRUEBAS = RAIZ / "tests" / "pipelines" / "training_pipeline"
FEATURES = RAIZ / "data" / "04_feature" / "corazon_features.parquet"
CHECKS = RAIZ / "data" / "08_reporting" / "checks_separacion.csv"

sys.path.insert(0, str(RAIZ / "src"))

from pipelines.training_pipeline import train_pipeline as tp  # noqa: E402

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 170)
pd.set_option("display.max_colwidth", 90)

print(f"Raíz del proyecto: {RAIZ}")
print(f"Script           : {SCRIPT.relative_to(RAIZ)}")

## 2. La función independiente

El requerimiento pide una función testeable e independiente. `validar_separacion_train_test`
recibe los cuatro conjuntos y devuelve el resultado: no escribe archivos, no lanza
excepciones y no depende del resto del pipeline. Se puede usar sobre cualquier partición.

El enunciado la nombra `validate_train_test_split()`; como el resto del código está en
español, ese es un alias del mismo objeto.

In [ ]:
print(
    f"Alias en inglés apunta a la misma función: {tp.validate_train_test_split is tp.validar_separacion_train_test}"
)
print()
print("Umbrales configurados:")
print(f"  tolerancia de estratificación : {tp.TOLERANCIA_ESTRATIFICACION:.0%}")
print(f"  tolerancia de proporción      : {tp.TOLERANCIA_PROPORCION:.0%}")
print(f"  alfa del test KS              : {tp.ALFA_DISTRIBUCION}")
print(f"  tolerancia de nulos           : {tp.TOLERANCIA_NULOS:.0%}")
print(f"  filas mínimas por conjunto    : {tp.MIN_FILAS_CONJUNTO}")

Sobre el alfa del test de Kolmogorov-Smirnov: se usa 0.01 y no el 0.05 habitual porque
se comparan 26 columnas a la vez. Con 0.05 cabría esperar más de un "atributo
significativamente distinto" por puro azar, y el check se volvería ruido que nadie mira.

## 3. La separación real del proyecto

Se aplican las nueve comprobaciones a la partición que produce el pipeline.

In [ ]:
atributos, objetivo = tp.leer_features(FEATURES)
x_train, x_test, y_train, y_test = tp.separar_train_test(atributos, objetivo)

resultado = tp.validar_separacion_train_test(x_train, x_test, y_train, y_test)

print(f"{resultado.resumen()}\n")
resultado.tabla()

### La advertencia que apareció

Ocho comprobaciones en verde y una advertencia real: la columna `max_hr` tiene una
diferencia del 10.4 % en su proporción de faltantes entre train y test, justo por encima
de la tolerancia del 10 %.

No es fuga ni un error de código: es el azar de repartir 480 filas con un 19 % de nulos
en esa columna. Pero merece quedar registrado, porque el imputador se ajusta con la
mediana del train y va a rellenar en el test más huecos de los que vio al aprender. Con
un umbral de tolerancia más laxo esto habría pasado desapercibido; ese es justamente el
tipo de detalle que estos checks existen para sacar a la luz.

In [ ]:
comparacion_nulos = pd.DataFrame(
    {
        "% nulos train": x_train.isna().mean(),
        "% nulos test": x_test.isna().mean(),
    }
)
comparacion_nulos["diferencia"] = (
    comparacion_nulos["% nulos train"] - comparacion_nulos["% nulos test"]
).abs()

comparacion_nulos.sort_values("diferencia", ascending=False).head(6).style.format("{:.1%}")

## 4. Comportamiento ante separaciones defectuosas

Cada comprobación se prueba rompiendo deliberadamente la partición correcta.

In [ ]:
def evaluar_caso(descripcion: str, particion: tuple, comprobacion: str) -> dict:
    """Aplica los checks a una partición alterada y reporta la comprobación de interés.

    `particion` es la tupla (x_train, x_test, y_train, y_test).
    """
    salida = tp.validar_separacion_train_test(*particion)
    encontrada = next(c for c in salida.comprobaciones if c.nombre == comprobacion)
    return {
        "caso": descripcion,
        "comprobación": comprobacion,
        "severidad": encontrada.severidad,
        "¿bloquea?": "sí" if not salida.valida else "no",
        "detalle": encontrada.detalle[:70],
    }

### Errores: fuga de información

Los tres casos que invalidan la evaluación por completo.

In [ ]:
# 1. La misma fila, con el mismo índice, en ambos conjuntos
fuga_indices = evaluar_caso(
    "3 filas de train copiadas al test",
    (x_train, pd.concat([x_test, x_train.head(3)]), y_train, pd.concat([y_test, y_train.head(3)])),
    "índices disjuntos",
)

# 2. La misma fila con OTRO índice: comparar índices no la detectaría
copia = x_train.head(3).copy()
copia.index = [999_001, 999_002, 999_003]
etiquetas = y_train.head(3).copy()
etiquetas.index = copia.index
fuga_duplicados = evaluar_caso(
    "3 filas duplicadas con índices nuevos",
    (x_train, pd.concat([x_test, copia]), y_train, pd.concat([y_test, etiquetas])),
    "filas duplicadas entre conjuntos",
)

# 3. Una clase entera ausente del test
solo_sanos = y_test[y_test == 0]
clase_ausente = evaluar_caso(
    "test sin ningún paciente enfermo",
    (x_train, x_test.loc[solo_sanos.index], y_train, solo_sanos),
    "cobertura de clases",
)

pd.DataFrame([fuga_indices, fuga_duplicados, clase_ausente])

El segundo caso es el que justifica la comprobación por huella de fila. Comparar
índices es lo primero que uno escribe, y no basta: si el dataset original traía el
mismo paciente dos veces y una copia cayó en cada conjunto, los índices son distintos
pero el modelo ya vio la respuesta. Por eso se compara también el contenido de la fila.

### Advertencias: representatividad

Estos casos no invalidan nada, pero cambian cómo hay que leer las métricas.

In [ ]:
# Reparto muy distinto del configurado
proporcion = evaluar_caso(
    "test recortado a 25 filas",
    (x_train, x_test.head(25), y_train, y_test.head(25)),
    "proporción de test",
)

# Prevalencia sesgada
sesgado = pd.concat([y_test[y_test == 1], y_test[y_test == 0].head(3)])
estratificacion = evaluar_caso(
    "test con casi sólo enfermos",
    (x_train, x_test.loc[sesgado.index], y_train, sesgado),
    "estratificación",
)

# Un atributo desplazado: el test pasa a ser de otra población
desplazado = x_test.copy()
desplazado["age"] = desplazado["age"] + 40
distribucion = evaluar_caso(
    "test con 40 años más de edad",
    (x_train, desplazado, y_train, y_test),
    "distribución de atributos (KS)",
)

# Faltantes desbalanceados
con_huecos = x_test.copy()
con_huecos.loc[con_huecos.index[: len(con_huecos) // 2], "chol"] = np.nan
nulos = evaluar_caso(
    "la mitad del test sin colesterol",
    (x_train, con_huecos, y_train, y_test),
    "faltantes comparables",
)

pd.DataFrame([proporcion, estratificacion, distribucion, nulos])

Nótese la columna `¿bloquea?`: en los cuatro casos es "no". Una advertencia informa,
no interrumpe. La decisión de qué hacer con ella es de quien lee el reporte.

## 5. Error controlado y ausencia de modelo

La comprobación decisiva del requerimiento: cuando la separación falla, el pipeline se
detiene con un mensaje claro y **no persiste ningún modelo**.

Aquí se aprovecha la advertencia real del proyecto: con `--estricto`, la diferencia de
faltantes en `max_hr` pasa a contar como error y aborta la ejecución. Se lanza sobre un
directorio temporal vacío para ver exactamente qué queda escrito.

In [ ]:
temporal = Path(tempfile.mkdtemp())

estricto = subprocess.run(  # noqa: S603
    [
        sys.executable,
        str(SCRIPT),
        "--estricto",
        "--modelo-salida",
        str(temporal / "modelo.joblib"),
        "--artefacto",
        str(temporal / "artefacto.joblib"),
        "--predicciones",
        str(temporal / "predicciones.csv"),
        "--metricas",
        str(temporal / "metricas.csv"),
        "--manifiesto",
        str(temporal / "manifiesto.json"),
        "--checks-split",
        str(temporal / "checks_separacion.csv"),
    ],
    capture_output=True,
    text=True,
    check=False,
)

print(estricto.stderr[-1200:])
print(f"Código de salida: {estricto.returncode}")
print(f"Archivos escritos: {sorted(p.name for p in temporal.iterdir())}")

Sólo queda `checks_separacion.csv`, que es precisamente lo que se quiere conservar: el
informe de por qué se abortó. Ningún `.joblib`, ninguna métrica. El orden en
`ejecutar_pipeline` lo garantiza estructuralmente — los checks corren antes de construir
el modelo, no en paralelo ni después.

El mensaje nombra la comprobación que falló, el valor concreto y el modo en que se
ejecutó. No lleva traza de Python porque el problema no está en el código: está en cómo
quedó repartido el dataset.

## 6. Ejecución normal

Sin `--estricto`, la advertencia se registra, el entrenamiento continúa y el resultado
de los checks queda guardado junto a las métricas.

In [ ]:
normal = subprocess.run(  # noqa: S603
    [sys.executable, str(SCRIPT)],
    capture_output=True,
    text=True,
    check=False,
)

lineas_split = [ln for ln in normal.stderr.splitlines() if "[split]" in ln]
print("\n".join(lineas_split))
print(f"\nCódigo de salida: {normal.returncode}")

In [ ]:
print("Resultados persistidos en data/08_reporting/checks_separacion.csv:\n")
pd.read_csv(CHECKS)

## 7. Pruebas unitarias

Cubren casos válidos e inválidos: para cada una de las nueve comprobaciones hay un test
que pasa y uno que falla, más los tests de la política (advertencias toleradas, modo
estricto) y dos de integración que verifican que ante fuga no se guarda ningún modelo.

In [ ]:
pruebas = subprocess.run(  # noqa: S603
    [
        sys.executable,
        "-m",
        "pytest",
        str(PRUEBAS),
        "-v",
        "--no-header",
        "-k",
        "separacion or fuga or split or estricto or duplicadas or clase or columnas "
        "or distribucion or faltantes or proporcion or estratificacion or alias",
    ],
    cwd=RAIZ,
    capture_output=True,
    text=True,
    check=False,
)

print(pruebas.stdout[-6000:])
print(f"Código de salida: {pruebas.returncode}")

## 8. Conclusiones

El requerimiento queda cubierto:

| Requisito | Dónde |
|---|---|
| Resultados de las validaciones | `ResultadoSeparacion.tabla()` y `data/08_reporting/checks_separacion.csv` |
| Advertencia clara | `logger.warning` por comprobación + resumen en el manifiesto |
| Error controlado | `ErrorDeSeparacion`, hereda de `ErrorDeValidacion`: sin traza y sin persistir |
| Función testeable e independiente | `validar_separacion_train_test` (alias `validate_train_test_split`) |
| Evitar fuga de información | índices disjuntos + huella de fila + cobertura de clases |
| Distribución esperada | proporción, estratificación, Kolmogorov-Smirnov y patrón de nulos |

Tres decisiones que vale la pena defender:

1. **Separar el cálculo de la decisión.** `validar_separacion_train_test` sólo mide;
   `aplicar_resultado_separacion` decide qué es fatal. Así la función de medida se puede
   usar para inspeccionar cualquier partición sin que aborte nada, y la política queda en
   un único sitio, configurable con `--estricto`.

2. **Dos severidades, no una.** Tratar todo como error obligaría a bajar los umbrales
   hasta volverlos inútiles, porque cualquier partición real tiene alguna asimetría.
   Tratar todo como advertencia dejaría pasar la fuga. La distinción entre "esto invalida
   la evaluación" y "esto cambia cómo la interpretas" es la que hace el check accionable.

3. **Comparar contenido, no sólo índices.** El check de huella de fila es el que atrapa
   la fuga que sobrevive a un `train_test_split` correcto: duplicados que ya venían en el
   origen. En este proyecto el feature pipeline los elimina antes, así que hoy no
   dispara — pero si mañana entra un dataset nuevo con duplicados, salta aquí en vez de
   convertirse en un F1 sospechosamente alto que nadie sabe explicar.